<a href="https://colab.research.google.com/github/Giriran25/TCMB/blob/main/NLm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


In [2]:
!git clone https://github.com/Giriran25/TCMB.git
%cd TCMB

Cloning into 'TCMB'...
remote: Enumerating objects: 43, done.
remote: Counting objects: 100% (43/43), done.
remote: Compressing objects: 100% (38/38), done.
remote: Total 43 (delta 3), reused 43 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (43/43), 1.44 MiB | 1.83 MiB/s, done.
Resolving deltas: 100% (3/3), done.
/content/TCMB


In [3]:
!pip install datasets transformers sentencepiece huggingface_hub pandas numpy scipy scikit-learn matplotlib tqdm regex opencc-python-reimplemented accelerate

In [4]:
!pwd
!find . -maxdepth 2 -type f | sort

/content/TCMB
./baselines/language_id_baseline.py
./baselines/static_xlmr.py
./evaluation/classification_metrics.py
./evaluation/profiling.py
./evaluation/statistical_analysis.py
./evaluation/tokenization_metrics.py
./.git/config
./.git/description
./.git/HEAD
./.gitignore
./.git/index
./.git/packed-refs
./notebooks/baseline_analysis.ipynb
./README.md
./requirements.txt
./scripts/audit_dataset_overlap.py
./scripts/collect_calibration_pilot.py
./scripts/diagnose_calibration_quality.py
./scripts/download_datasets.py
./scripts/inspect_calibration_samples.py
./scripts/inspect_calibration_sources.py
./scripts/inspect_chinese_variants.py
./scripts/validate_calibration_pilot.py
./tcmb/anchor_constraints.py
./tcmb/budget_predictor.py
./tcmb/composer.py
./tcmb/__init__.py
./tcmb/pair_scorer.py
./tcmb/typological_features.py
./training/leave_one_out.py
./training/train_composer.py
./training/train_tcmb.py


In [5]:
import torch
import transformers
import datasets
import pandas
import numpy
import sklearn
import sentencepiece

print("All imports successful!")
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

All imports successful!
PyTorch: 2.11.0+cu130
CUDA: True
GPU: Tesla T4


In [7]:
from huggingface_hub import notebook_login

notebook_login()

In [8]:
from huggingface_hub import whoami

print(whoami())

{'type': 'user', 'id': '69d54b62bd2bdf61c0bf335c', 'name': 'giriranrk28', 'fullname': 'Ranjith Kumar G', 'canPay': False, 'billingMode': 'prepaid', 'periodEnd': 1796083200, 'isPro': False, 'avatarUrl': '/avatars/0bce33f412760f7bc73c3470af3d479f.svg', 'orgs': [], 'auth': {'type': 'oauth', 'expiresAt': '2026-12-05T08:28:26.000Z'}}


In [9]:
!sed -n '1,240p' scripts/download_datasets.py

from pathlib import Path
from datasets import load_dataset
import shutil

DATASETS = {
    "flores_plus": {
        "repository": "openlanguagedata/flores_plus",
        "languages": [
            "eng_Latn",
            "tam_Taml",
            "tel_Telu",
            "kan_Knda",
            "mal_Mlym",
            "hin_Deva",
            "cmn_Hans",
            "jpn_Jpan"
        ]
    },

    "sib200": {
        "repository": "Davlan/sib200",
        "languages": [
            "eng_Latn",
            "tam_Taml",
            "tel_Telu",
            "kan_Knda",
            "mal_Mlym",
            "hin_Deva",
            "zho_Hans",
            "jpn_Jpan"
        ]
    }
}

BASE_DIR = Path("data")


def download_dataset(dataset_name, repository, language):

    output_dir = BASE_DIR / dataset_name / language
    manifest = output_dir / "dataset_dict.json"

    if manifest.exists():
        print(f"[EXISTS] {dataset_name}/{language}")
        return True

    print(f"\nDownloading {datas

In [10]:
from scripts.download_datasets import download_dataset, DATASETS

languages = DATASETS["flores_plus"]["languages"]

for language in languages:
    download_dataset(
        dataset_name="flores_plus",
        repository=DATASETS["flores_plus"]["repository"],
        language=language
    )

README.md:   0%|          | 0.00/75.5k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/233 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/228 [00:00<?, ?it/s]

eng_Latn.jsonl:   0%|          | 0.00/423k [00:00<?, ?B/s]

eng_Latn.jsonl:   0%|          | 0.00/440k [00:00<?, ?B/s]

Generating dev split: 0 examples [00:00, ? examples/s]

... (63 outputs left out)

Generating devtest split: 0 examples [00:00, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/997 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1012 [00:00<?, ? examples/s]

[SAVED] data/flores_plus/jpn_Jpan
  dev: 997 examples
  devtest: 1012 examples


In [11]:
from pathlib import Path
from datasets import load_from_disk

base = Path("data/flores_plus")

for language_dir in sorted(base.iterdir()):
    if language_dir.is_dir() and not language_dir.name.endswith(".partial"):
        dataset = load_from_disk(str(language_dir))

        print(f"\nLanguage: {language_dir.name}")

        for split, data in dataset.items():
            print(f"{split}: {len(data)} examples")


Language: cmn_Hans
dev: 997 examples
devtest: 1012 examples

Language: eng_Latn
dev: 997 examples
devtest: 1012 examples

Language: hin_Deva
dev: 997 examples
devtest: 1012 examples

Language: jpn_Jpan
dev: 997 examples
devtest: 1012 examples

Language: kan_Knda
dev: 997 examples
devtest: 1012 examples

Language: mal_Mlym
dev: 997 examples
devtest: 1012 examples

Language: tam_Taml
dev: 997 examples
devtest: 1012 examples

Language: tel_Telu
dev: 997 examples
devtest: 1012 examples


In [14]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [15]:
!python -m baselines.static_xlmr --help

usage: static_xlmr.py [-h] [--split {dev,devtest}] [--batch-size BATCH_SIZE]

Evaluate static XLM-R tokenization across eight FLORES+ languages.

options:
  -h, --help            show this help message and exit
  --split {dev,devtest}
                        FLORES+ dataset split.
  --batch-size BATCH_SIZE
                        Number of sentences processed per batch.


In [16]:
!ls -lh results/tables
!ls -lh results/figures

total 3.1M
-rw-r--r-- 1 root root 1.8K Nov  5 08:24 calibration_pilot_summary.csv
-rw-r--r-- 1 root root  975 Nov  5 08:24 calibration_pilot_validation.csv
-rw-r--r-- 1 root root 440K Nov  5 08:24 calibration_quality_sample_metadata.csv
-rw-r--r-- 1 root root 1.9K Nov  5 08:24 calibration_quality_summary.csv
-rw-r--r-- 1 root root 1.9M Nov  5 08:24 dataset_overlap_details.csv
-rw-r--r-- 1 root root 6.4K Nov  5 08:24 dataset_overlap_summary.csv
-rw-r--r-- 1 root root 790K Nov  5 08:24 static_xlmr_sentence_metrics_dev.csv
-rw-r--r-- 1 root root 2.2K Nov  5 08:24 static_xlmr_summary_dev.csv
total 684K
-rw-r--r-- 1 root root 145K Nov  5 08:24 static_xlmr_content_tokens_per_grapheme_dev.png
-rw-r--r-- 1 root root 128K Nov  5 08:24 static_xlmr_content_whitespace_fertility_dev.png
-rw-r--r-- 1 root root 142K Nov  5 08:24 static_xlmr_marker_share_dev.png
-rw-r--r-- 1 root root 140K Nov  5 08:24 static_xlmr_token_to_grapheme_dev.png
-rw-r--r-- 1 root root 122K Nov  5 08:24 static_xlmr_whitespac

In [17]:
!grep -nE "load_from_disk|load_dataset|from_pretrained|cuda|device|save_to_disk|output_dir|DATA_DIR" baselines/static_xlmr.py

28:from datasets import load_from_disk
116:    dataset_dict = load_from_disk(str(dataset_path))
493:    tokenizer = AutoTokenizer.from_pretrained(


In [18]:
!find data/flores_plus -maxdepth 2 -type f | head -30

data/flores_plus/eng_Latn/dataset_dict.json
data/flores_plus/hin_Deva/dataset_dict.json
data/flores_plus/kan_Knda/dataset_dict.json
data/flores_plus/jpn_Jpan/dataset_dict.json
data/flores_plus/tel_Telu/dataset_dict.json
data/flores_plus/cmn_Hans/dataset_dict.json
data/flores_plus/mal_Mlym/dataset_dict.json
data/flores_plus/tam_Taml/dataset_dict.json


In [19]:
!sed -n '90,135p' baselines/static_xlmr.py
!sed -n '470,560p' baselines/static_xlmr.py
!tail -100 baselines/static_xlmr.py

        Dataset split: dev or devtest.

    batch_size:
        Number of sentences processed per batch.

    Returns
    -------
    tuple:
        Language summary and sentence-level records.
    """

    dataset_path = (
        project_root
        / "data"
        / "flores_plus"
        / language_code
    )

    # Check whether the dataset exists.
    if not dataset_path.exists():
        raise FileNotFoundError(
            f"Dataset folder not found: {dataset_path}"
        )

    print(f"\nLoading dataset: {language_code}")

    dataset_dict = load_from_disk(str(dataset_path))

    # Check split availability.
    if split_name not in dataset_dict:
        raise ValueError(
            f"Split '{split_name}' not found for {language_code}. "
            f"Available splits: {list(dataset_dict.keys())}"
        )

    dataset = dataset_dict[split_name]

    # Check required columns.
    required_columns = {"id", "text"}

    missing_columns = required_columns - set(
        datas

In [20]:
from pathlib import Path
import shutil

backup = Path("results/backup_before_colab_rerun")
backup.mkdir(parents=True, exist_ok=True)

for folder in ["tables", "figures"]:
    source = Path("results") / folder
    if source.exists():
        shutil.copytree(
            source,
            backup / folder,
            dirs_exist_ok=True
        )

print("Existing results backed up successfully!")

Existing results backed up successfully!


In [21]:
!python -m baselines.static_xlmr --split dev --batch-size 32

TCMB - STATIC XLM-R TOKENIZATION BASELINE
Model: FacebookAI/xlm-roberta-base
Dataset split: dev
Batch size: 32

Loading XLM-R tokenizer...
config.json: 100% 615/615 [00:00<00:00, 2.02MB/s]
tokenizer_config.json: 100% 25.0/25.0 [00:00<00:00, 86.0kB/s]
sentencepiece.bpe.model: 100% 5.07M/5.07M [00:00<00:00, 5.30MB/s]
tokenizer.json: 100% 9.10M/9.10M [00:00<00:00, 21.1MB/s]
Tokenizer loaded successfully.
Vocabulary size: 250002
Standalone marker: ['▁']
Standalone marker ID: 6

Loading dataset: eng_Latn

Evaluating English (eng_Latn) - dev
English: 100% 32/32 [00:00<00:00, 178.41it/s]
Completed English: 997 sentences

Loading dataset: tam_Taml

Evaluating Tamil (tam_Taml) - dev
Tamil: 100% 32/32 [00:00<00:00, 86.56it/s]
Completed Tamil: 997 sentences

Loading dataset: tel_Telu

Evaluating Telugu (tel_Telu) - dev
Telugu: 100% 32/32 [00:00<00:00, 75.88it/s]
Completed Telugu: 997 sentences

Loading dataset: kan_Knda

Evaluating Kannada (kan_Knda) - dev
Kannada: 100% 32/32 [00:00<00:00, 74.99i

In [22]:
import torch
from transformers import AutoTokenizer, AutoModel

MODEL_NAME = "FacebookAI/xlm-roberta-base"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
print("GPU:", torch.cuda.get_device_name(0))

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModel.from_pretrained(MODEL_NAME)
model = model.to(device)
model.eval()

text = "This is a test sentence for TCMB."

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(device)

with torch.no_grad():
    # Extract pretrained token embeddings
    embeddings = model.get_input_embeddings()(
        inputs["input_ids"]
    )

    # Run XLM-R using inputs_embeds
    outputs = model(
        inputs_embeds=embeddings,
        attention_mask=inputs["attention_mask"]
    )

print("\nGPU verification successful!")
print("Input IDs shape:", inputs["input_ids"].shape)
print("Embedding shape:", embeddings.shape)
print("Output shape:", outputs.last_hidden_state.shape)
print("Embedding dimension:", embeddings.shape[-1])
print(
    "GPU memory allocated:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

Device: cuda
GPU: Tesla T4


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



GPU verification successful!
Input IDs shape: torch.Size([1, 11])
Embedding shape: torch.Size([1, 11, 768])
Output shape: torch.Size([1, 11, 768])
Embedding dimension: 768
GPU memory allocated: 1.04 GB


In [23]:
# Get original token embeddings
input_ids = inputs["input_ids"]
attention_mask = inputs["attention_mask"]

# Identify special tokens
special_mask = tokenizer.get_special_tokens_mask(
    input_ids[0].tolist(),
    already_has_special_tokens=True
)

# Find two adjacent non-special tokens
merge_position = next(
    i for i in range(1, len(special_mask) - 2)
    if special_mask[i] == 0
    and special_mask[i + 1] == 0
)

# Simple mean composition for testing only
merged_embedding = (
    embeddings[:, merge_position:merge_position + 1, :]
    + embeddings[:, merge_position + 1:merge_position + 2, :]
) / 2

# Create a shorter embedding sequence
short_embeddings = torch.cat([
    embeddings[:, :merge_position, :],
    merged_embedding,
    embeddings[:, merge_position + 2:, :]
], dim=1)

# Update attention mask
short_attention_mask = torch.ones(
    short_embeddings.shape[:2],
    dtype=torch.long,
    device=device
)

# Pass the shorter sequence to XLM-R
with torch.no_grad():
    merged_outputs = model(
        inputs_embeds=short_embeddings,
        attention_mask=short_attention_mask
    )

print("Original sequence length:", embeddings.shape[1])
print("Merged sequence length:", short_embeddings.shape[1])
print("Original embedding shape:", embeddings.shape)
print("Merged embedding shape:", short_embeddings.shape)
print("Transformer output shape:", merged_outputs.last_hidden_state.shape)

Original sequence length: 11
Merged sequence length: 10
Original embedding shape: torch.Size([1, 11, 768])
Merged embedding shape: torch.Size([1, 10, 768])
Transformer output shape: torch.Size([1, 10, 768])


In [24]:
!cat tcmb/composer.py
!cat tcmb/pair_scorer.py
!cat tcmb/budget_predictor.py

In [25]:
from pathlib import Path

files = [
    "tcmb/composer.py",
    "tcmb/pair_scorer.py",
    "tcmb/budget_predictor.py",
]

for filename in files:
    print("\n" + "=" * 70)
    print(filename)
    print("=" * 70)

    path = Path(filename)
    if path.exists():
        print(path.read_text())
    else:
        print("File not found!")


tcmb/composer.py


tcmb/pair_scorer.py


tcmb/budget_predictor.py



In [26]:
from pathlib import Path

files = [
    "tcmb/__init__.py",
    "tcmb/typological_features.py",
    "tcmb/anchor_constraints.py",
    "training/train_tcmb.py",
    "training/train_composer.py",
    "training/leave_one_out.py",
    "requirements.txt",
]

for filename in files:
    path = Path(filename)

    print("\n" + "=" * 75)
    print(filename)
    print("=" * 75)

    if path.exists():
        content = path.read_text()
        print(content if content.strip() else "[EMPTY FILE]")
    else:
        print("[FILE NOT FOUND]")


tcmb/__init__.py
[EMPTY FILE]

tcmb/typological_features.py
[EMPTY FILE]

tcmb/anchor_constraints.py
[EMPTY FILE]

training/train_tcmb.py
[EMPTY FILE]

training/train_composer.py
[EMPTY FILE]

training/leave_one_out.py
[EMPTY FILE]

requirements.txt
[EMPTY FILE]


In [27]:
from pathlib import Path

print("PROJECT DATA INVENTORY")
print("=" * 65)

for folder_name in ["data", "calibration", "results", "tcmb", "training"]:
    folder = Path(folder_name)

    print(f"\n{folder_name}/")

    if not folder.exists():
        print("  [Folder not found]")
        continue

    for path in sorted(folder.rglob("*")):
        if path.is_file():
            print(f"  {path} ({path.stat().st_size / 1024:.1f} KB)")

PROJECT DATA INVENTORY

data/
  data/flores_plus/cmn_Hans/dataset_dict.json (0.0 KB)
  data/flores_plus/cmn_Hans/dev/data-00000-of-00001.arrow (286.9 KB)
  data/flores_plus/cmn_Hans/dev/dataset_info.json (1.5 KB)
  data/flores_plus/cmn_Hans/dev/state.json (0.2 KB)
  data/flores_plus/cmn_Hans/devtest/data-00000-of-00001.arrow (300.0 KB)
  data/flores_plus/cmn_Hans/devtest/dataset_info.json (1.5 KB)
  data/flores_plus/cmn_Hans/devtest/state.json (0.2 KB)
  data/flores_plus/eng_Latn/dataset_dict.json (0.0 KB)
  data/flores_plus/eng_Latn/dev/data-00000-of-00001.arrow (294.8 KB)
  data/flores_plus/eng_Latn/dev/dataset_info.json (1.5 KB)
  data/flores_plus/eng_Latn/dev/state.json (0.2 KB)
  data/flores_plus/eng_Latn/devtest/data-00000-of-00001.arrow (311.1 KB)
  data/flores_plus/eng_Latn/devtest/dataset_info.json (1.5 KB)
  data/flores_plus/eng_Latn/devtest/state.json (0.2 KB)
  data/flores_plus/hin_Deva/dataset_dict.json (0.0 KB)
  data/flores_plus/hin_Deva/dev/data-00000-of-00001.arrow (48

In [28]:
import pandas as pd

path = Path("results/tables/calibration_pilot_summary.csv")

if path.exists():
    df = pd.read_csv(path)
    print("Calibration summary columns:")
    print(df.columns.tolist())
    display(df.head(10))
else:
    print("Calibration pilot summary not found.")

Calibration summary columns:
['language_code', 'language', 'source', 'configuration', 'target_tokens', 'actual_tokens', 'target_reached', 'accepted_documents', 'scanned_documents', 'unique_domains', 'short_documents_rejected', 'wrong_language_rejected', 'wrong_script_rejected', 'low_script_ratio_rejected', 'duplicate_documents_rejected', 'evaluation_overlaps_rejected', 'oversized_documents_rejected', 'invalid_documents_rejected', 'output_file']


,language_code,language,source,configuration,target_tokens,actual_tokens,target_reached,accepted_documents,scanned_documents,unique_domains,short_documents_rejected,wrong_language_rejected,wrong_script_rejected,low_script_ratio_rejected,duplicate_documents_rejected,evaluation_overlaps_rejected,oversized_documents_rejected,invalid_documents_rejected,output_file
0,eng_Latn,English,HuggingFaceFW/fineweb,sample-10BT,100000,100182,True,125,129,124,4,0,0,0,0,0,0,0,C:\Users\Ranjith Kumar G\Documents\TCMB\data\c...
1,tam_Taml,Tamil,HuggingFaceFW/fineweb-2,tam_Taml,100000,100125,True,110,112,101,1,0,0,0,0,0,1,0,C:\Users\Ranjith Kumar G\Documents\TCMB\data\c...
2,tel_Telu,Telugu,HuggingFaceFW/fineweb-2,tel_Telu,100000,101866,True,100,103,76,0,0,0,3,0,0,0,0,C:\Users\Ranjith Kumar G\Documents\TCMB\data\c...
3,kan_Knda,Kannada,HuggingFaceFW/fineweb-2,kan_Knda,100000,101433,True,135,136,77,0,0,0,1,0,0,0,0,C:\Users\Ranjith Kumar G\Documents\TCMB\data\c...
4,mal_Mlym,Malayalam,HuggingFaceFW/fineweb-2,mal_Mlym,100000,100697,True,151,153,101,0,0,0,2,0,0,0,0,C:\Users\Ranjith Kumar G\Documents\TCMB\data\c...
5,hin_Deva,Hindi,HuggingFaceFW/fineweb-2,hin_Deva,100000,100259,True,142,151,121,4,0,0,4,0,0,1,0,C:\Users\Ranjith Kumar G\Documents\TCMB\data\c...
6,cmn_Hans,Chinese,HuggingFaceFW/fineweb-2,cmn_Hani,100000,100576,True,112,135,110,21,0,0,1,0,0,1,0,C:\Users\Ranjith Kumar G\Documents\TCMB\data\c...
7,jpn_Jpan,Japanese,HuggingFaceFW/fineweb-2,jpn_Jpan,100000,100246,True,129,153,125,20,0,0,1,0,0,3,0,C:\Users\Ranjith Kumar G\Documents\TCMB\data\c...


In [29]:
import pandas as pd
from pathlib import Path

df = pd.read_csv("results/tables/calibration_pilot_summary.csv")

print("ORIGINAL CALIBRATION FILE PATHS")
print("=" * 70)

for _, row in df.iterrows():
    print(f"\n{row['language']}:")
    print(row["output_file"])

print("\n\nCALIBRATION COLLECTION SCRIPT")
print("=" * 70)

path = Path("scripts/collect_calibration_pilot.py")
lines = path.read_text().splitlines()

for i, line in enumerate(lines, 1):
    if any(word in line.lower() for word in [
        "output_file", "output_dir", "save", "jsonl",
        "write_text", "open(", "calibration"
    ]):
        print(f"{i}: {line}")

ORIGINAL CALIBRATION FILE PATHS

English:
C:\Users\Ranjith Kumar G\Documents\TCMB\data\calibration_pilot\eng_Latn\documents.jsonl

Tamil:
C:\Users\Ranjith Kumar G\Documents\TCMB\data\calibration_pilot\tam_Taml\documents.jsonl

Telugu:
C:\Users\Ranjith Kumar G\Documents\TCMB\data\calibration_pilot\tel_Telu\documents.jsonl

Kannada:
C:\Users\Ranjith Kumar G\Documents\TCMB\data\calibration_pilot\kan_Knda\documents.jsonl

Malayalam:
C:\Users\Ranjith Kumar G\Documents\TCMB\data\calibration_pilot\mal_Mlym\documents.jsonl

Hindi:
C:\Users\Ranjith Kumar G\Documents\TCMB\data\calibration_pilot\hin_Deva\documents.jsonl

Chinese:
C:\Users\Ranjith Kumar G\Documents\TCMB\data\calibration_pilot\cmn_Hans\documents.jsonl

Japanese:
C:\Users\Ranjith Kumar G\Documents\TCMB\data\calibration_pilot\jpn_Jpan\documents.jsonl


CALIBRATION COLLECTION SCRIPT
2: TCMB - Calibration Corpus Pilot Collection
22:     data/calibration_pilot/<language_code>/documents.jsonl
23:     results/tables/calibration_pilot_summ

In [ ]:
%cd /content/TCMB

!python scripts/collect_calibration_pilot.py

/content/TCMB
TCMB - 100K TOKEN CALIBRATION PILOT
Model: FacebookAI/xlm-roberta-base
Target per language: 100000
Total target: 800000
Random seed: 42

COLLECTING: English
Loading FLORES+ exclusion index: eng_Latn
FLORES+ exclusion sentences: 2008
README.md: 100% 44.3k/44.3k [00:00<00:00, 50.7MB/s]
Resolving data files: 100% 27468/27468 [00:01<00:00, 17314.48it/s]
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1156 > 512). Running this sequence through the model will result in indexing errors
Accepted: 25 | Tokens: 17,024
Accepted: 50 | Tokens: 32,444
Accepted: 75 | Tokens: 44,617
Accepted: 100 | Tokens: 71,351
Accepted: 125 | Tokens: 91,340

Collection summary:
language_code: eng_Latn
language: English
source: HuggingFaceFW/fineweb
configuration: sample-10BT
target_tokens: 100000
actual_tokens: 100284
target_reached: True
accepted_documents: 140
scanned_documents: 146
unique_domains: 140
short_documents_rejected: 6
wron